In [1]:
import sys
import pyspark

In [2]:
import os

#Utilizamos Java11 para que sea compatible con la version de Spark que utilizamos

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [3]:
import pandas as pd

# Cargar cada CSV en una tabla independiente
df_datos = pd.read_csv("data/dataset.csv")
df_categorias = pd.read_csv("data/categorias_auxiliar.csv")

# Comprobar filas y columnas
print("Dataset:", df_datos.shape)
print("Categorías:", df_categorias.shape)

Dataset: (1000000, 12)
Categorías: (81, 2)


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("prueba1")
    .getOrCreate()
)

print("Master:", spark.sparkContext.master)
spark.range(5).show()

Master: spark://spark-master:7077
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [5]:
ruta = "hdfs://namenode:9000/donostiarrak/raw/dataset.csv"

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta)
)

df_raw.printSchema()
df_raw.show(5)

root
 |-- email_id: string (nullable = true)
 |-- texto: string (nullable = true)
 |-- subcategoria: string (nullable = true)
 |-- gestor_principal: string (nullable = true)
 |-- gestor_secundario: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- idioma: string (nullable = true)
 |-- perfil_escritura: string (nullable = true)
 |-- es_ambiguo: boolean (nullable = true)
 |-- tiene_multiples_incidencias: boolean (nullable = true)
 |-- familia_escenario: string (nullable = true)
 |-- split: string (nullable = true)

+----------------+--------------------+------------------+--------------------+-----------------+-----------+----------+----------------+----------+---------------------------+------------------+----------+
|        email_id|               texto|      subcategoria|    gestor_principal|gestor_secundario|       zona|    idioma|perfil_escritura|es_ambiguo|tiene_multiples_incidencias| familia_escenario|     split|
+----------------+--------------------+------------

In [6]:
print("Columnas:", df_raw.columns)
df_raw.printSchema()
df_raw.show(5, truncate=False)

Columnas: ['email_id', 'texto', 'subcategoria', 'gestor_principal', 'gestor_secundario', 'zona', 'idioma', 'perfil_escritura', 'es_ambiguo', 'tiene_multiples_incidencias', 'familia_escenario', 'split']
root
 |-- email_id: string (nullable = true)
 |-- texto: string (nullable = true)
 |-- subcategoria: string (nullable = true)
 |-- gestor_principal: string (nullable = true)
 |-- gestor_secundario: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- idioma: string (nullable = true)
 |-- perfil_escritura: string (nullable = true)
 |-- es_ambiguo: boolean (nullable = true)
 |-- tiene_multiples_incidencias: boolean (nullable = true)
 |-- familia_escenario: string (nullable = true)
 |-- split: string (nullable = true)

+----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------+-----

In [7]:
ruta_categorias = (
    "hdfs://namenode:9000/donostiarrak/raw/categorias_auxiliar.csv"
)

df_categorias = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_categorias)
)

df_categorias.printSchema()
df_categorias.show(10, truncate=False)

root
 |-- categoria: string (nullable = true)
 |-- subcategoria: string (nullable = true)

+---------+------------------------+
|categoria|subcategoria            |
+---------+------------------------+
|CULTURA  |Actividades culturales  |
|CULTURA  |Equipamientos culturales|
|CULTURA  |San Telmo               |
|CULTURA  |Victoria Eugenia        |
|DEPORTE  |Actividades deportivas  |
|DEPORTE  |Cursos deportivos       |
|DEPORTE  |Equipamientos deportivos|
|DEPORTE  |Instalaciones deportivas|
|EDUCACIÓN|Actividades educativas  |
|EDUCACIÓN|Centros educativos      |
+---------+------------------------+
only showing top 10 rows



In [18]:
from pyspark.sql import functions as F

ruta_raw = "hdfs://namenode:9000/donostiarrak/raw/dataset.csv"
ruta_aux = "hdfs://namenode:9000/donostiarrak/raw/categorias_auxiliar.csv"

df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_raw)
)

df_categorias = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_aux)
)

# Comprobar los nombres exactos antes del join
print("Dataset:", df_raw.columns)
print("Auxiliar:", df_categorias.columns)

# Conservar todas las filas del dataset y añadir su categoria
df_enriquecido = df_raw.join(
    df_categorias.select("subcategoria", "categoria"),
    on="subcategoria",
    how="left"
)

# Comprobar si alguna subcategoria no encontró categoria
sin_categoria = df_enriquecido.filter(
    F.col("categoria").isNull()
).count()

print("Filas sin categoria asignada:", sin_categoria)
df_enriquecido.select("subcategoria", "categoria").show(10, truncate=False)

# Unificar nombres equivalentes usando nombres en castellano
equivalencias_zona = {
    "Alde Zaharra": "Parte Vieja",
    "Mirakruz-Bidebieta": "Miracruz-Bidebieta",
    "Loiolako Erriberak": "Riberas de Loiola",
    "Erdialdea": "Centro",
}

df_enriquecido = df_enriquecido.replace(equivalencias_zona, subset=["zona"])
df_enriquecido.groupBy("zona").count().orderBy("zona").show(100, truncate=False)

Dataset: ['email_id', 'texto', 'subcategoria', 'gestor_principal', 'gestor_secundario', 'zona', 'idioma', 'perfil_escritura', 'es_ambiguo', 'tiene_multiples_incidencias', 'familia_escenario', 'split']
Auxiliar: ['categoria', 'subcategoria']
Filas sin categoria asignada: 0
+----------------------------+-----------------------+
|subcategoria                |categoria              |
+----------------------------+-----------------------+
|Malos olores                |MEDIO AMBIENTE         |
|Certificados                |FINANCIERA             |
|Vivienda                    |URBANISMO              |
|Animales                    |SEGURIDAD Y CONVIVENCIA|
|Obras en viviendas          |URBANISMO              |
|Equipamientos deportivos    |DEPORTE                |
|Aparcamientos               |MOVILIDAD              |
|Calidad del aire            |MEDIO AMBIENTE         |
|Disciplina urbanística      |URBANISMO              |
|Plagas - Limpieza y residuos|LIMPIEZA Y RESIDUOS    |
+-----------

In [19]:
from pyspark.sql import functions as F

# Rutas de los archivos originales en HDFS
ruta_raw = "hdfs://namenode:9000/donostiarrak/raw/dataset.csv"
ruta_auxiliar = (
    "hdfs://namenode:9000/donostiarrak/raw/categorias_auxiliar.csv"
)

# Leer los CSV desde HDFS
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_raw)
)

df_categorias = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_auxiliar)
)

# Comprobar los nombres de las columnas
print("Columnas del dataset:", df_raw.columns)
print("Columnas de la tabla auxiliar:", df_categorias.columns)

# Comprobar que cada subcategoria aparece una sola vez en la tabla auxiliar
subcategorias_repetidas = (
    df_categorias
    .groupBy("subcategoria")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if subcategorias_repetidas > 0:
    raise ValueError(
        f"Hay {subcategorias_repetidas} subcategorias repetidas "
        "en la tabla auxiliar."
    )

# Añadir la categoria a cada fila usando subcategoria como clave
df_enriquecido = df_raw.join(
    df_categorias.select("subcategoria", "categoria"),
    on="subcategoria",
    how="left"
)

# Comprobar que todas las filas encontraron categoria
filas_sin_categoria = (
    df_enriquecido
    .filter(F.col("categoria").isNull())
    .count()
)

print("Filas sin categoria asignada:", filas_sin_categoria)

if filas_sin_categoria > 0:
    raise ValueError(
        "Hay filas sin categoria asignada. Revisa las subcategorias."
    )

# Comprobar que el join conservó el número de filas
filas_raw = df_raw.count()
filas_enriquecidas = df_enriquecido.count()

print("Filas originales:", filas_raw)
print("Filas tras añadir categoria:", filas_enriquecidas)

if filas_raw != filas_enriquecidas:
    raise ValueError(
        "El join ha cambiado el número de filas. Revisa la tabla auxiliar."
    )

# Guardar como Parquet, creando una particion por categoria
ruta_curada = (
    "hdfs://namenode:9000/donostiarrak/"
    "curada/dataset_por_categoria"
)

(
    df_enriquecido.write
    .mode("overwrite")
    .partitionBy("categoria")
    .parquet(ruta_curada)
)

print("Parquet guardado en:", ruta_curada)

Columnas del dataset: ['email_id', 'texto', 'subcategoria', 'gestor_principal', 'gestor_secundario', 'zona', 'idioma', 'perfil_escritura', 'es_ambiguo', 'tiene_multiples_incidencias', 'familia_escenario', 'split']
Columnas de la tabla auxiliar: ['categoria', 'subcategoria']
Filas sin categoria asignada: 0
Filas originales: 1000000
Filas tras añadir categoria: 1000000
Parquet guardado en: hdfs://namenode:9000/donostiarrak/curada/dataset_por_categoria


In [20]:
ruta_curada = (
    "hdfs://namenode:9000/donostiarrak/"
    "curada/dataset_por_categoria"
)

df_curada = spark.read.parquet(ruta_curada)

df_curada.printSchema()
df_curada.show(5, truncate=False)

root
 |-- subcategoria: string (nullable = true)
 |-- email_id: string (nullable = true)
 |-- texto: string (nullable = true)
 |-- gestor_principal: string (nullable = true)
 |-- gestor_secundario: string (nullable = true)
 |-- zona: string (nullable = true)
 |-- idioma: string (nullable = true)
 |-- perfil_escritura: string (nullable = true)
 |-- es_ambiguo: boolean (nullable = true)
 |-- tiene_multiples_incidencias: boolean (nullable = true)
 |-- familia_escenario: string (nullable = true)
 |-- split: string (nullable = true)
 |-- categoria: string (nullable = true)

+-----------------------+----------------+---------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+-----------------+-----------+----------+----------------+----------+---------------------------+-----------------------+----------+---------+
|subcategoria           |email_id        |texto                     

In [21]:
df_curada.select("categoria").distinct().orderBy("categoria").show(
    50, truncate=False
)

+-----------------------+
|categoria              |
+-----------------------+
|CULTURA                |
|DEPORTE                |
|EDUCACIÓN              |
|ESPACIO PÚBLICO        |
|EUSKERA                |
|FINANCIERA             |
|LIMPIEZA Y RESIDUOS    |
|MEDIO AMBIENTE         |
|MOVILIDAD              |
|OBRAS                  |
|SEGURIDAD Y CONVIVENCIA|
|SERVICIOS SOCIALES     |
|TRANSPORTE             |
|TURISMO                |
|URBANISMO              |
+-----------------------+



In [22]:
print("Filas:", df_curada.count())

Filas: 1000000


In [23]:
#### PARA ML ########

from pyspark.sql import functions as F

ruta_raw = "hdfs://namenode:9000/donostiarrak/raw/dataset.csv"
ruta_auxiliar = (
    "hdfs://namenode:9000/donostiarrak/raw/categorias_auxiliar.csv"
)

# Leer los CSV originales desde HDFS
df_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_raw)
)

df_categorias = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(ruta_auxiliar)
)

# Comprobar que cada subcategoria tiene una sola categoria
subcategorias_repetidas = (
    df_categorias
    .groupBy("subcategoria")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if subcategorias_repetidas > 0:
    raise ValueError(
        f"Hay {subcategorias_repetidas} subcategorias repetidas."
    )

# Unir y conservar solo las columnas para ML
df_ml = (
    df_raw
    .join(
        df_categorias.select("subcategoria", "categoria"),
        on="subcategoria",
        how="left"
    )
    .select(
        F.col("texto").alias("rawtext"),
        "categoria",
        "subcategoria",
        "zona",
        "split"
    )
)

# Comprobar que todas las filas tienen categoria
filas_sin_categoria = (
    df_ml
    .filter(F.col("categoria").isNull())
    .count()
)

if filas_sin_categoria > 0:
    raise ValueError(
        f"Hay {filas_sin_categoria} filas sin categoria."
    )

# Guardar como Parquet, particionado por categoria
ruta_parquet_ml = (
    "hdfs://namenode:9000/donostiarrak/curada/parquet_ML"
)

(
    df_ml.write
    .mode("overwrite")
    .partitionBy("categoria")
    .parquet(ruta_parquet_ml)
)

print("Columnas:", df_ml.columns)
print("Parquet ML guardado en:", ruta_parquet_ml)
df_ml.show(5, truncate=False)

Columnas: ['rawtext', 'categoria', 'subcategoria', 'zona', 'split']
Parquet ML guardado en: hdfs://namenode:9000/donostiarrak/curada/parquet_ML
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------+------------------+-----------+----------+
|rawtext                                                                                                                                                                                                                     |categoria              |subcategoria      |zona       |split     |
+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------+------------------+-----------+-

In [24]:
import time
from pyspark.sql import functions as F

ruta_csv = "hdfs://namenode:9000/donostiarrak/raw/dataset.csv"
ruta_parquet = (
    "hdfs://namenode:9000/donostiarrak/curada/dataset_por_categoria"
)

columnas_originales = [
    "email_id",
    "texto",
    "subcategoria",
    "gestor_principal",
    "gestor_secundario",
    "zona",
    "idioma",
    "perfil_escritura",
    "es_ambiguo",
    "tiene_multiples_incidencias",
    "familia_escenario",
    "split",
]

def medir_lectura(formato, ruta):
    inicio = time.perf_counter()

    if formato == "csv":
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv(ruta)
        )
    else:
        df = spark.read.parquet(ruta).select(*columnas_originales)

    # Fuerza a Spark a leer los valores, no solo a crear el DataFrame
    expresiones = [
        F.sum(
            F.length(F.coalesce(F.col(c).cast("string"), F.lit("")))
        ).alias(f"col_{i}")
        for i, c in enumerate(columnas_originales)
    ]
    df.select(*expresiones).collect()

    return time.perf_counter() - inicio

print(f"CSV:     {medir_lectura('csv', ruta_csv):.2f} s")
print(f"Parquet: {medir_lectura('parquet', ruta_parquet):.2f} s")

CSV:     6.97 s
Parquet: 6.11 s
